# TC-PhysXAI: Physics-Grounded XAI for Tropical Cyclone Track Prediction

**Author:** Nguyen Hong Gam  
**Paper:** TC-XFlow + TC-PhysXAI (ESWA 2025 + Extension)  
**Environment:** Kaggle GPU (P100/T4)

---

## Overview

| Module | Contribution | Description |
|--------|-------------|-------------|
| Module 1 | **C1** | Temporal attribution via climatological perturbation (ΔCRPS per 6h window) |
| Module 2 | **C2** | Integrated Gradients on FNO3D + physical object detection |
| Module 2 | **C3** | H-mean-F fidelity metric = 0.4·P + 0.4·R + 0.2·U |
| Module 3 | **M3** | DTW K-medoids scenario clustering + latent analogue retrieval |
| C4 | **C4** | FM integration-path attribution A_j = ∫₀¹ α_j dt |
| C5 | **C5** | Variance decomposition: V_total = V_noise + V_context |
| C6 | **C6** | FM counterfactual via projected gradient on context space |
| **NEW** | **C7** | Segment-level temporal XAI + AOPC faithfulness (XAI4Traj-inspired) |
| **NEW** | **C8** | Track Feature Attribution: speed/curvature/heading → ERA5 channel/window drivers |

**Note:** This notebook uses a pre-trained TC-XFlow checkpoint. No training required.

---

## 1. Environment Setup

In [ ]:
# ── 1.1 Mount TC-XFlow training architecture ──────────────────────────────────
import sys, os

# TC-XFlow source (provides Model.flow_matching_model)
TC_XFLOW_DIR = "/kaggle/working/Tropical_Typhoon_Architecture"
# TC-PhysXAI source
TC_PHYSXAI_DIR = "/kaggle/working/TC-PhysXAI"

for d in [TC_XFLOW_DIR, TC_PHYSXAI_DIR]:
    if d not in sys.path:
        sys.path.insert(0, d)

print("Python path configured.")
print(f"TC-XFlow:   {TC_XFLOW_DIR}")
print(f"TC-PhysXAI: {TC_PHYSXAI_DIR}")

In [ ]:
# ── 1.2 Copy TC-PhysXAI from Kaggle dataset input ────────────────────────────
!cp -r /kaggle/input/datasets/kaggle1234uitvn/tc-physxai/TC-PhysXAI /kaggle/working/TC-PhysXAI
print("TC-PhysXAI copied.")

In [ ]:
# ── 1.3 Verify imports ────────────────────────────────────────────────────────
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

# TC-PhysXAI modules
from PhysXAI import (
    TemporalAttributor,
    SpatialAttributor,
    AnalogueXAI,
    FMIntegrationAttribution,
    VarianceDecomposition,
    FMCounterfactual,
    SegmentPerturbationXAI,
    TrackFeatureAttributor,   # C8
    TrackFeatureExtractor,    # C8
    ProbabilisticFidelity,
    MeteorologyInterpreter,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load TC-XFlow Checkpoint (No Training Required)

In [ ]:
# ── 2.1 Load the pre-trained TC-XFlow model ───────────────────────────────────
# Path to the existing checkpoint (from previous training run)
CHECKPOINT_PATH = "/kaggle/working/runs/fm_v3_ot_fixed/seed0/best_model.pt"

from Model.flow_matching_model import TCFlowMatching

# Load checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

# Build model with saved config
model_cfg = ckpt.get("model_config", {})
model = TCFlowMatching(**model_cfg).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"Model loaded from: {CHECKPOINT_PATH}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {n_trainable:,}")

In [ ]:
# ── 2.2 Load test dataset ─────────────────────────────────────────────────────
DATASET_ROOT = "/kaggle/input/datasets/kaggle1234uitvn/tc-ofm"

from Data.dataset import TCDataset
from torch.utils.data import DataLoader
from Data.collate import seq_collate

test_dataset = TCDataset(
    root=DATASET_ROOT,
    split="test",
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=seq_collate,
)

print(f"Test storms: {len(test_dataset)}")

# Collect first N_STORMS batches
N_STORMS = 20
all_batches = []
for i, batch in enumerate(test_loader):
    if i >= N_STORMS:
        break
    all_batches.append(batch)

print(f"Loaded {len(all_batches)} test batches.")

# Quick sanity check — batch_list structure:
# [0] obs_traj   [T_obs, B, 2]   (normalised lat/lon)
# [1] pred_traj  [T_pred, B, 2]  (ground truth)
# [7] obs_Me     [T_obs, B, 4]   (lat, lon, MinPressure, MaxWind)
# [11] img_obs   [B, 13, T_obs, 81, 81]  (ERA5 fields)
b0 = all_batches[0]
print(f"obs_traj shape:  {b0[0].shape}")
print(f"pred_traj shape: {b0[1].shape}")
print(f"img_obs shape:   {b0[11].shape}")

## 3. XAI Analysis — Single Storm Demo

Run all XAI modules on Storm #0 to demonstrate outputs.

In [ ]:
# ── 3.1 Choose analysis storm ─────────────────────────────────────────────────
STORM_IDX = 0           # which batch to analyse
K_ENSEMBLE = 30         # number of ensemble members

batch_list = all_batches[STORM_IDX]
print(f"Analysing Storm #{STORM_IDX}")
print(f"Ensemble size K={K_ENSEMBLE}")

### 3.1 Module 1 — C1: Temporal Attribution (ΔCRPS)

In [ ]:
# C1: Which 6h observation window matters most for the forecast?
ta = TemporalAttributor(model, DEVICE)
c1_result = ta.compute(batch_list, K=K_ENSEMBLE, verbose=True)

print("\n" + "="*60)
print("C1 TEMPORAL ATTRIBUTION")
print("="*60)
print(c1_result['physical_interpretation'])
print(f"\nDominant window: t−{(c1_result['dominant_step']+1)*6}h")
print(f"Max ΔCRPS:       {max(c1_result['delta_crps']):.4f}")

In [ ]:
# Visualise C1 attribution
fig, ax = plt.subplots(figsize=(10, 4))

delta_crps = c1_result['delta_crps']   # [T_obs]
T_obs = len(delta_crps)
time_labels = [f"t−{(T_obs - i)*6}h" for i in range(T_obs)]

colors = ['#d62728' if v == max(delta_crps) else '#1f77b4' for v in delta_crps]
ax.bar(time_labels, delta_crps, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_title(f"C1: Temporal Attribution — Storm #{STORM_IDX}\n"
             "ΔCRPS per 6h window (red = most critical)", fontsize=12)
ax.set_ylabel("ΔCRPS (higher = more informative)")
ax.set_xlabel("Observation window")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f"/kaggle/working/TC-PhysXAI/results/c1_temporal_storm{STORM_IDX}.png", dpi=150)
plt.show()
print("C1 figure saved.")

### 3.2 Module 2 — C2: Spatial IG + Physical Object Detection

In [ ]:
# C2: Which ERA5 spatial regions drive the velocity field prediction?
sa = SpatialAttributor(model, DEVICE)
c2_result = sa.compute(batch_list, n_ig_steps=30, verbose=True)

print("\n" + "="*60)
print("C2 SPATIAL ATTRIBUTION")
print("="*60)
print(c2_result['physical_interpretation'])
print(f"\nTop 3 ERA5 channels: {c2_result['top_channels'][:3]}")
print(f"Physical objects detected: {len(c2_result['detected_objects'])}")
for obj in c2_result['detected_objects'][:5]:
    print(f"  - {obj.label} | channel={obj.channel_idx} | "
          f"score={obj.attribution_score:.4f}")

In [ ]:
# Visualise C2 — channel attribution bar chart
CH_NAMES = [
    "GPH_200", "GPH_500", "GPH_850", "GPH_925",
    "U_200",   "U_500",   "U_850",   "U_925",
    "V_200",   "V_500",   "V_850",   "V_925",
    "SST",
]

ch_attr = c2_result['channel_attribution']   # [13]

fig, ax = plt.subplots(figsize=(12, 4))
colors_c2 = plt.cm.RdBu_r(np.linspace(0.1, 0.9, 13))
ax.bar(CH_NAMES, ch_attr, color=colors_c2, edgecolor='black', linewidth=0.5)
ax.set_title(f"C2: ERA5 Channel Attribution (IG) — Storm #{STORM_IDX}", fontsize=12)
ax.set_ylabel("Attribution Score (normalised IG)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f"/kaggle/working/TC-PhysXAI/results/c2_spatial_storm{STORM_IDX}.png", dpi=150)
plt.show()
print("C2 figure saved.")

### 3.3 Module 3 — DTW Ensemble Scenario Clustering

In [ ]:
# M3: Cluster ensemble trajectories into physical scenarios
ax_xai = AnalogueXAI(model, DEVICE, n_clusters=3)
m3_result = ax_xai.compute(batch_list, K=K_ENSEMBLE, verbose=True)

print("\n" + "="*60)
print("MODULE 3: ENSEMBLE SCENARIO DECOMPOSITION")
print("="*60)
print(m3_result['cluster_summary'])
print(f"\nEntropy (normalised): {m3_result['uncertainty_entropy']:.3f}")
print(f"Dominant scenario:    {m3_result['dominant_scenario']}")

In [ ]:
# Visualise M3 — ensemble spaghetti plot with cluster colours
from PhysXAI.utils import denorm_latlon

_, _, all_traj_m3 = model.sample(batch_list, num_ensemble=K_ENSEMBLE,
                                  use_speed_calibration=True, return_xai=False)
# all_traj_m3: [K, T_pred, B, 2]

clusters = m3_result['clusters']
cluster_colors = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e']

fig, ax = plt.subplots(figsize=(10, 8))

# Plot ensemble members coloured by cluster
labels_arr = np.zeros(K_ENSEMBLE, dtype=int)
for ci, cl in enumerate(clusters):
    for mi in cl.member_indices:
        labels_arr[mi] = ci

for k in range(K_ENSEMBLE):
    traj_k = all_traj_m3[k, :, 0, :].cpu().numpy()  # [T_pred, 2] storm 0
    ci = labels_arr[k]
    color = cluster_colors[ci % len(cluster_colors)]
    ax.plot(traj_k[:, 1], traj_k[:, 0], color=color, alpha=0.25, linewidth=0.8)

# Plot cluster centroids
for ci, cl in enumerate(clusters):
    cent = cl.centroid   # [T_pred, 2]
    color = cluster_colors[ci % len(cluster_colors)]
    ax.plot(cent[:, 1], cent[:, 0], color=color, linewidth=2.5,
            label=f"{cl.physical_label} ({cl.probability:.0%})")

# Ground truth
gt = batch_list[1][:, 0, :].cpu().numpy()  # [T_pred, 2]
ax.plot(gt[:, 1], gt[:, 0], 'k-', linewidth=2.5, marker='o',
        markersize=4, label='Ground Truth')

ax.set_title(f"M3: Ensemble Scenario Decomposition — Storm #{STORM_IDX}\n"
             f"Entropy={m3_result['uncertainty_entropy']:.2f}", fontsize=12)
ax.set_xlabel("Normalised Longitude")
ax.set_ylabel("Normalised Latitude")
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f"/kaggle/working/TC-PhysXAI/results/m3_scenarios_storm{STORM_IDX}.png", dpi=150)
plt.show()
print("M3 figure saved.")

### 3.4 C4 — FM Integration-Path Attribution

In [ ]:
# C4: How does each ERA5 channel influence the FM velocity field along the ODE path?
fm_attr = FMIntegrationAttribution(model, DEVICE, ddim_steps=20)
c4_result = fm_attr.compute(batch_list, verbose=True)

print("\n" + "="*60)
print("C4 FM INTEGRATION-PATH ATTRIBUTION")
print("="*60)
print(c4_result.physical_interpretation)
print(f"\nDominant channels: {c4_result.dominant_channels}")

In [ ]:
# Visualise C4 — ODE path attribution curve + channel bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# ODE time attribution
n_steps = len(c4_result.ode_time_attribution)
t_vals  = np.linspace(0, 1, n_steps)
ax1.plot(t_vals, c4_result.ode_time_attribution, 'b-', linewidth=2)
ax1.fill_between(t_vals, c4_result.ode_time_attribution, alpha=0.3)
ax1.set_xlabel("ODE time t ∈ [0,1]")
ax1.set_ylabel("Attribution (mean |∂v/∂c|)")
ax1.set_title("C4: Attribution along ODE path")

# Channel attribution
ch_attr_c4 = c4_result.channel_attribution
bars = ax2.bar(CH_NAMES, ch_attr_c4, color='steelblue', edgecolor='black', linewidth=0.5)
top3 = np.argsort(ch_attr_c4)[::-1][:3]
for i in top3:
    bars[i].set_color('#d62728')
ax2.set_title("C4: ERA5 Channel Attribution (FM path)")
ax2.set_ylabel("Attribution Score")
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')

plt.suptitle(f"C4: FM Integration-Path Attribution — Storm #{STORM_IDX}", fontsize=13)
plt.tight_layout()
plt.savefig(f"/kaggle/working/TC-PhysXAI/results/c4_fm_attr_storm{STORM_IDX}.png", dpi=150)
plt.show()
print("C4 figure saved.")

### 3.5 C5 — Noise-Context Variance Decomposition

In [ ]:
# C5: Is uncertainty from stochastic noise (z0) or from synoptic context ambiguity?
vd = VarianceDecomposition(model, DEVICE, n_context_perturb=10)
c5_result = vd.compute(batch_list, K=K_ENSEMBLE, verbose=True)

print("\n" + "="*60)
print("C5 VARIANCE DECOMPOSITION")
print("="*60)
print(c5_result.physical_interpretation)
print(f"\nV_noise:   {c5_result.var_noise:.6f} ({c5_result.frac_noise:.1%})")
print(f"V_context: {c5_result.var_context:.6f} ({c5_result.frac_context:.1%})")
print(f"Pred. horizon: step {c5_result.predictability_horizon} "
      f"= t+{c5_result.predictability_horizon*6}h")

In [ ]:
# Visualise C5 — stacked variance decomposition over forecast horizon
T_pred = len(c5_result.per_horizon_noise)
times  = [f"t+{(i+1)*6}h" for i in range(T_pred)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.stackplot(
    range(T_pred),
    c5_result.per_horizon_noise,
    c5_result.per_horizon_context,
    labels=['V_noise (stochastic z₀)', 'V_context (synoptic ambiguity)'],
    colors=['#1f77b4', '#ff7f0e'],
    alpha=0.75,
)
ax.axvline(c5_result.predictability_horizon, color='red', linestyle='--',
           label=f'Predictability horizon (t+{c5_result.predictability_horizon*6}h)')
ax.set_xticks(range(T_pred))
ax.set_xticklabels(times, rotation=30)
ax.set_xlabel("Forecast horizon")
ax.set_ylabel("Variance (normalised units)")
ax.set_title(f"C5: Noise-Context Variance Decomposition — Storm #{STORM_IDX}\n"
             f"V_noise={c5_result.frac_noise:.0%} | V_context={c5_result.frac_context:.0%}",
             fontsize=12)
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f"/kaggle/working/TC-PhysXAI/results/c5_variance_storm{STORM_IDX}.png", dpi=150)
plt.show()
print("C5 figure saved.")

### 3.6 C6 — FM Counterfactual Generation

In [ ]:
# C6: What atmospheric change would cause this typhoon to recurve?
cf = FMCounterfactual(model, DEVICE, ddim_steps=20, max_iter=80)
c6_result = cf.compute(batch_list, cf_type="RECURVATURE", verbose=True)

print("\n" + "="*60)
print("C6 COUNTERFACTUAL EXPLANATION")
print("="*60)
print(c6_result.physical_explanation)
print(f"\nContext change norm: {c6_result.context_change_norm:.2%}")
print(f"Max track shift:     {c6_result.track_delta_km:.0f} km")

In [ ]:
# Visualise C6 — original vs counterfactual vs target track
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Track comparison
orig = c6_result.original_track       # [T_pred, 2]
cf_t = c6_result.counterfactual_track  # [T_pred, 2]
tgt  = c6_result.target_track         # [T_pred, 2]
gt   = batch_list[1][:, 0, :].cpu().numpy()

ax1.plot(orig[:, 1], orig[:, 0], 'b-o', markersize=5, linewidth=2, label='Original prediction')
ax1.plot(cf_t[:, 1], cf_t[:, 0], 'r-o', markersize=5, linewidth=2, label='Counterfactual')
ax1.plot(tgt[:, 1],  tgt[:, 0],  'g--', linewidth=1.5, alpha=0.7, label='CF target')
ax1.plot(gt[:, 1],   gt[:, 0],   'k-s', markersize=4, linewidth=2, label='Ground truth')
ax1.set_title(f"C6: Counterfactual Tracks (RECURVATURE)\nContext Δ={c6_result.context_change_norm:.1%}", fontsize=11)
ax1.set_xlabel("Normalised Longitude")
ax1.set_ylabel("Normalised Latitude")
ax1.legend(fontsize=8)

# Channel delta
ch_delta = c6_result.channel_delta   # [13]
ax2.bar(CH_NAMES, ch_delta, color='#9467bd', edgecolor='black', linewidth=0.5)
ax2.set_title("C6: ERA5 Channel Changes Required for Counterfactual", fontsize=11)
ax2.set_ylabel("Relative channel perturbation")
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')

plt.suptitle(f"C6: FM Counterfactual — Storm #{STORM_IDX}", fontsize=13)
plt.tight_layout()
plt.savefig(f"/kaggle/working/TC-PhysXAI/results/c6_counterfactual_storm{STORM_IDX}.png", dpi=150)
plt.show()
print("C6 figure saved.")

### 3.7 C7 (NEW) — Segment-Level Temporal XAI + AOPC Faithfulness

Inspired by **XAI4Traj** (Towards Explainable Trajectory Classification, Neurocomputing 2025).

In [ ]:
# C7: Which OBSERVATION PHASE drives the forecast?
#   Phase A: REMOTE PRECURSOR  (t−48h to t−24h)
#   Phase B: SYNOPTIC EVOLUTION (t−24h to t−12h)
#   Phase C: IMMEDIATE STEERING (t−12h to t−0h)

sp = SegmentPerturbationXAI(model, DEVICE, K=20)
c7_result = sp.compute(batch_list, verbose=True)

print("\n" + "="*60)
print("C7 SEGMENT-LEVEL TEMPORAL XAI")
print("="*60)
print(c7_result.physical_interpretation)
print(f"\nDominant phase:      {c7_result.dominant_phase}")
print(f"Uncertainty source:  {c7_result.uncertainty_source}")
print(f"AOPC faithfulness:   {c7_result.aopc_faithfulness:+.4f}")

In [ ]:
# Visualise C7 — phase attribution bar + AOPC curves
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# Phase attribution
ax1 = fig.add_subplot(gs[0, 0])
phase_names_short = ['A\nREMOTE', 'B\nSYNOPTIC', 'C\nIMMEDIATE']
phase_deltas = [r.delta_crps for r in sorted(
    c7_result.phase_results,
    key=lambda r: ['A_REMOTE_PRECURSOR','B_SYNOPTIC_EVOLUTION','C_IMMEDIATE_STEERING'].index(r.phase_name)
)]
bar_colors = ['#d62728' if v == max(phase_deltas) else '#1f77b4' for v in phase_deltas]
ax1.bar(phase_names_short, phase_deltas, color=bar_colors, edgecolor='black')
ax1.axhline(0, color='k', linewidth=0.5)
ax1.set_title("Phase Attribution (ΔCRPS)")
ax1.set_ylabel("ΔCRPS (higher = more informative)")

# AOPC MoRF curve
ax2 = fig.add_subplot(gs[0, 1])
n_steps = len(c7_result.aopc_morf_curve)
x_steps = range(n_steps)
ax2.plot(x_steps, c7_result.aopc_morf_curve, 'r-o', label='MoRF (most→least)', linewidth=2)
ax2.plot(x_steps, c7_result.aopc_lerf_curve, 'b-s', label='LeRF (least→most)', linewidth=2)
ax2.fill_between(x_steps, c7_result.aopc_morf_curve, c7_result.aopc_lerf_curve,
                 alpha=0.2, color='purple', label='AOPC gap')
ax2.set_xticks(x_steps)
ax2.set_xticklabels(['Full', 'Phase 1', 'Phase 2', 'All masked'][:n_steps])
ax2.set_title(f"AOPC Curves\nFaithfulness={c7_result.aopc_faithfulness:+.3f}")
ax2.set_ylabel("CRPS")
ax2.legend(fontsize=8)

# Relative attribution pie
ax3 = fig.add_subplot(gs[0, 2])
rel_attrs = [max(r.relative_attribution, 0.01) for r in sorted(
    c7_result.phase_results,
    key=lambda r: ['A_REMOTE_PRECURSOR','B_SYNOPTIC_EVOLUTION','C_IMMEDIATE_STEERING'].index(r.phase_name)
)]
ax3.pie(rel_attrs, labels=['A\nRemote', 'B\nSynoptic', 'C\nImmediate'],
        colors=['#1f77b4','#ff7f0e','#2ca02c'],
        autopct='%1.0f%%', startangle=90, textprops={'fontsize': 9})
ax3.set_title("Relative Phase Attribution")

plt.suptitle(f"C7: Segment-Level Temporal XAI — Storm #{STORM_IDX}", fontsize=13)
plt.savefig(f"/kaggle/working/TC-PhysXAI/results/c7_segment_storm{STORM_IDX}.png", dpi=150)
plt.show()
print("C7 figure saved.")

### 3.8 C8 (NEW) — Track Feature Attribution

**C8** decomposes predicted track features (speed, curvature, recurvature probability,
heading change, displacement, trajectory spread, bifurcation index) into contributions
from each ERA5 channel (GPH/U/V at 4 pressure levels + SST) and observation window.

Method: channel-zeroing attribution + spatiotemporal mean perturbation baseline,
with bootstrap CIs and MoRF-AOPC faithfulness verification.

In [ ]:
# C8: Which ERA5 channels drive each predicted track feature?
tfa = TrackFeatureAttributor(
    model, DEVICE,
    K=K_ENSEMBLE,
    n_bootstrap=20,   # 20 bootstrap iterations for CIs
)

c8_result = tfa.compute(batch_list, verbose=True)

print("\n" + "="*60)
print("C8 TRACK FEATURE ATTRIBUTION")
print("="*60)
print(c8_result.physical_interpretation)
print(f"\nTop channel-feature pair: {c8_result.top_channel} → {c8_result.top_feature}")
print(f"AOPC faithfulness:        {c8_result.aopc_faithfulness:+.4f}")
print(f"Bootstrap CI coverage:    {c8_result.n_significant}/{c8_result.n_total} pairs significant")

In [ ]:
# Visualise C8 — attribution heatmap (features × channels)
import matplotlib.patches as mpatches

CH_NAMES = [
    "GPH_200", "GPH_500", "GPH_850", "GPH_925",
    "U_200",   "U_500",   "U_850",   "U_925",
    "V_200",   "V_500",   "V_850",   "V_925",
    "SST",
]
FEAT_NAMES = [
    "speed", "curvature", "recurvature_prob", "heading_change",
    "displacement", "spread", "bifurcation_idx",
]

attr_mat = c8_result.attribution_matrix   # [7, 13]
sig_mat  = c8_result.significance_mask    # [7, 13] bool

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
ax = axes[0]
im = ax.imshow(attr_mat, aspect='auto', cmap='RdBu_r',
               vmin=-np.abs(attr_mat).max(), vmax=np.abs(attr_mat).max())
ax.set_xticks(range(13))
ax.set_xticklabels(CH_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(7))
ax.set_yticklabels(FEAT_NAMES, fontsize=9)
# Mark significant cells
for fi in range(7):
    for ci in range(13):
        if sig_mat[fi, ci]:
            ax.add_patch(mpatches.Rectangle(
                (ci - 0.5, fi - 0.5), 1, 1,
                fill=False, edgecolor='black', linewidth=1.5
            ))
plt.colorbar(im, ax=ax, label='Attribution (Δfeature)')
ax.set_title(f"C8: Feature × Channel Attribution\n(black border = bootstrap-significant, α=0.05)")

# Per-channel bar chart (sum over features)
ax2 = axes[1]
ch_total = np.abs(attr_mat).sum(axis=0)  # [13]
ax2.bar(CH_NAMES, ch_total, color='steelblue', edgecolor='black', linewidth=0.5)
ax2.set_title("C8: Total |Attribution| per ERA5 Channel")
ax2.set_ylabel("Sum |Δfeature| over all track features")
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=8)

plt.suptitle(f"C8: Track Feature Attribution — Storm #{STORM_IDX}\n"
             f"AOPC faithfulness={c8_result.aopc_faithfulness:+.3f}", fontsize=13)
plt.tight_layout()
plt.savefig(f"/kaggle/working/TC-PhysXAI/results/c8_feature_attr_storm{STORM_IDX}.png", dpi=150)
plt.show()
print("C8 figure saved.")

In [ ]:
# C8 — Window attribution: which 6h observation window matters most per feature?
window_attr = c8_result.window_attribution   # [7, T_obs]
T_obs = window_attr.shape[1]
time_labels_c8 = [f"t−{(T_obs-i)*6}h" for i in range(T_obs)]

fig, ax = plt.subplots(figsize=(11, 5))
for fi, fname in enumerate(FEAT_NAMES):
    ax.plot(time_labels_c8, window_attr[fi], marker='o', linewidth=1.5,
            label=fname, alpha=0.85)
ax.set_title(f"C8: Observation Window Attribution per Track Feature — Storm #{STORM_IDX}")
ax.set_xlabel("Observation window")
ax.set_ylabel("Attribution (Δfeature per masked window)")
ax.legend(fontsize=8, ncol=2, loc='upper left')
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f"/kaggle/working/TC-PhysXAI/results/c8_window_attr_storm{STORM_IDX}.png", dpi=150)
plt.show()
print("C8 window figure saved.")

### 3.8 Meteorological Bulletin — Integrated Physical Interpretation

In [ ]:
# Combine all XAI results into a physical bulletin
interp = MeteorologyInterpreter()

xai_bundle = {
    'c1': c1_result,
    'c2': c2_result,
    'm3': m3_result,
    'c4': c4_result,
    'c5': c5_result,
    'c6': c6_result,
    'c7': c7_result,
}

bulletin = interp.generate_bulletin(xai_bundle)
print(bulletin)

# Save bulletin
Path("/kaggle/working/TC-PhysXAI/results").mkdir(parents=True, exist_ok=True)
with open(f"/kaggle/working/TC-PhysXAI/results/bulletin_storm{STORM_IDX}.txt", 'w') as f:
    f.write(bulletin)
print("Bulletin saved.")

## 4. Batch Analysis — All N_STORMS

In [ ]:
# ── 4.1 Batch XAI analysis across all storms ─────────────────────────────────
import json

# Configuration
K_BATCH = 20           # smaller K for batch speed
OUTPUT_DIR = Path("/kaggle/working/TC-PhysXAI/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Initialise XAI modules once
ta_batch  = TemporalAttributor(model, DEVICE)
vd_batch  = VarianceDecomposition(model, DEVICE, n_context_perturb=8)
sp_batch  = SegmentPerturbationXAI(model, DEVICE, K=K_BATCH)

batch_results = []

for storm_i, bl in enumerate(all_batches):
    print(f"\n[{storm_i+1}/{N_STORMS}] Analysing storm {storm_i}...")
    try:
        # C1
        r_c1 = ta_batch.compute(bl, K=K_BATCH)
        # C5
        r_c5 = vd_batch.compute(bl, K=K_BATCH)
        # C7
        r_c7 = sp_batch.compute(bl)

        batch_results.append({
            'storm_idx': storm_i,
            'c1_dominant_step': int(r_c1['dominant_step']),
            'c1_max_delta_crps': float(max(r_c1['delta_crps'])),
            'c5_frac_noise': float(r_c5.frac_noise),
            'c5_frac_context': float(r_c5.frac_context),
            'c5_pred_horizon': int(r_c5.predictability_horizon),
            'c7_dominant_phase': r_c7.dominant_phase,
            'c7_uncertainty_source': r_c7.uncertainty_source,
            'c7_aopc_faithfulness': float(r_c7.aopc_faithfulness),
            'c7_crps_full': float(r_c7.crps_full),
        })
        print(f"  C1 dominant: t−{(r_c1['dominant_step']+1)*6}h | "
              f"C5 noise={r_c5.frac_noise:.0%} | "
              f"C7 phase={r_c7.dominant_phase.split('_')[0]} | "
              f"AOPC={r_c7.aopc_faithfulness:+.3f}")

    except Exception as e:
        print(f"  ERROR: {e}")
        continue

# Save JSON
with open(OUTPUT_DIR / "batch_xai_results.json", 'w') as f:
    json.dump(batch_results, f, indent=2)

print(f"\nBatch analysis complete. Results saved to {OUTPUT_DIR / 'batch_xai_results.json'}")

In [ ]:
# ── 4.2 Batch summary statistics ─────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(batch_results)
print("=" * 60)
print("BATCH XAI SUMMARY STATISTICS")
print("=" * 60)
print(df.describe().to_string())

# Uncertainty source distribution
print("\nC7 Uncertainty Source Distribution:")
print(df['c7_uncertainty_source'].value_counts().to_string())

print("\nC7 Dominant Phase Distribution:")
print(df['c7_dominant_phase'].value_counts().to_string())

print(f"\nMean AOPC Faithfulness: {df['c7_aopc_faithfulness'].mean():.4f}")
print(f"(> 0 = attribution is faithful, < 0 = counter-faithful)")

In [ ]:
# ── 4.3 Batch visualisation ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# C5 noise-context distribution
ax = axes[0, 0]
ax.scatter(df['c5_frac_noise'], df['c5_frac_context'],
           c=df['c7_aopc_faithfulness'], cmap='RdYlGn', s=80, edgecolor='black')
ax.axline((0.5, 0), slope=-1, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel("V_noise fraction")
ax.set_ylabel("V_context fraction")
ax.set_title("C5: Noise vs Context Variance\n(colour = AOPC faithfulness)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# C1 dominant step histogram
ax = axes[0, 1]
obs_steps = list(range(8))
counts = [sum(df['c1_dominant_step'] == s) for s in obs_steps]
time_labels_b = [f"t−{(8-s)*6}h" for s in obs_steps]
ax.bar(time_labels_b, counts, color='steelblue', edgecolor='black')
ax.set_title("C1: Distribution of Dominant Observation Window")
ax.set_ylabel("Number of storms")
plt.setp(ax.get_xticklabels(), rotation=30)

# C7 AOPC faithfulness distribution
ax = axes[1, 0]
ax.hist(df['c7_aopc_faithfulness'], bins=10, color='#2ca02c', edgecolor='black', alpha=0.8)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Faithfulness = 0')
ax.set_title("C7: AOPC Faithfulness Distribution")
ax.set_xlabel("AOPC faithfulness (MoRF − LeRF)")
ax.set_ylabel("Count")
ax.legend()

# C7 dominant phase distribution
ax = axes[1, 1]
phase_short = df['c7_uncertainty_source'].map({
    'A': 'Remote\nPrecursor',
    'B': 'Synoptic\nEvolution',
    'C': 'Immediate\nSteering',
    'MIXED': 'Mixed',
}).fillna('Other')
vc = phase_short.value_counts()
ax.bar(vc.index, vc.values,
       color=['#1f77b4','#ff7f0e','#2ca02c','#9467bd'][:len(vc)],
       edgecolor='black')
ax.set_title("C7: Dominant Uncertainty Source per Storm")
ax.set_ylabel("Number of storms")

plt.suptitle(f"TC-PhysXAI Batch Summary — {N_STORMS} Test Storms", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "batch_summary.png", dpi=150)
plt.show()
print("Batch summary saved.")

## 5. Experiments E1–E8

Run individual experiments separately or all at once.

In [ ]:
# ── E1: XAI Baseline Comparison ──────────────────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Experiments/E1_comparison_baselines.py \
    --checkpoint /kaggle/working/runs/fm_v3_ot_fixed/seed0/best_model.pt \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 30 --gpu 0

In [ ]:
# ── E2: Recurvature Attribution ───────────────────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Experiments/E2_recurvature_attribution.py \
    --checkpoint /kaggle/working/runs/fm_v3_ot_fixed/seed0/best_model.pt \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 30 --gpu 0

In [ ]:
# ── E3: Rapid Intensification & SST Attribution ───────────────────────────────
!python /kaggle/working/TC-PhysXAI/Experiments/E3_rapid_intensification.py \
    --checkpoint /kaggle/working/runs/fm_v3_ot_fixed/seed0/best_model.pt \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 15 --gpu 0

In [ ]:
# ── E4: Multi-Modal Consistency ────────────────────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Experiments/E4_multimodal_consistency.py \
    --checkpoint /kaggle/working/runs/fm_v3_ot_fixed/seed0/best_model.pt \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 20 --gpu 0

In [ ]:
# ── E5: H-mean-F Fidelity Benchmark (Ablation Study) ─────────────────────────
!python /kaggle/working/TC-PhysXAI/Experiments/E5_fidelity_benchmark.py \
    --checkpoint /kaggle/working/runs/fm_v3_ot_fixed/seed0/best_model.pt \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir /kaggle/working/TC-PhysXAI/results \
    --n_storms 15 --gpu 0

In [ ]:
# ── E6: Counterfactual Scenarios ──────────────────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Experiments/E6_counterfactual_scenarios.py \
    --checkpoint /kaggle/working/runs/fm_v3_ot_fixed/seed0/best_model.pt \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir /kaggle/working/TC-PhysXAI/results \
    --n_storms 15 --gpu 0

In [ ]:
# ── E7: Adaptive Segment-Level XAI (C7 Extended) ────────────────────────────
!python /kaggle/working/TC-PhysXAI/Experiments/E7_adaptive_segment_xai.py \
    --checkpoint /kaggle/working/runs/fm_v3_ot_fixed/seed0/best_model.pt \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 20 --gpu 0

In [ ]:
# ── E8: Track Feature Attribution (C8) + K-Sensitivity Ablation ─────────────
!python /kaggle/working/TC-PhysXAI/Experiments/E8_track_feature_attribution.py \
    --checkpoint /kaggle/working/runs/fm_v3_ot_fixed/seed0/best_model.pt \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 10 --gpu 0

In [ ]:
# ── ALL Experiments at once ────────────────────────────────────────────────────
# (Runs E1–E8 sequentially; takes ~45–90 min on T4)
!python /kaggle/working/TC-PhysXAI/Experiments/run_all_experiments.py \
    --checkpoint /kaggle/working/runs/fm_v3_ot_fixed/seed0/best_model.pt \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir /kaggle/working/TC-PhysXAI/results \
    --gpu 0 --n_storms 20 --K 30 \
    --experiments E1 E2 E3 E4 E5 E6 E7 E8

## 6. Full Pipeline (Single Command)

The `run_physxai.py` script runs all XAI modules on N storms and saves results.

In [ ]:
# ── Full analysis on 20 test storms ──────────────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Scripts/run_physxai.py \
    --checkpoint    /kaggle/working/runs/fm_v3_ot_fixed/seed0/best_model.pt \
    --dataset_root  /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir    /kaggle/working/TC-PhysXAI/results \
    --gpu           0 \
    --n_storms      20 \
    --K             30 \
    --run_experiments

In [ ]:
# ── Single storm quick test (storm index 0) ───────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Scripts/run_physxai.py \
    --checkpoint    /kaggle/working/runs/fm_v3_ot_fixed/seed0/best_model.pt \
    --dataset_root  /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir    /kaggle/working/TC-PhysXAI/results \
    --gpu           0 \
    --n_storms      1 \
    --K             10

## 7. Results Summary

Quick check on output files.

In [ ]:
# List all generated result files
import os

results_dir = "/kaggle/working/TC-PhysXAI/results"
print(f"Results in {results_dir}:")
for root, dirs, files in os.walk(results_dir):
    level = root.replace(results_dir, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        size_kb = os.path.getsize(os.path.join(root, f)) / 1024
        print(f'{indent}  {f} ({size_kb:.0f} KB)')

## 8. C7 Ablation — AOPC Faithfulness Comparison

Compare AOPC faithfulness across all test storms to validate that the
physical phase segmentation produces faithful attributions.

In [ ]:
# Reload batch results if needed
with open(OUTPUT_DIR / "batch_xai_results.json") as f:
    batch_results = json.load(f)

df = pd.DataFrame(batch_results)

# AOPC faithfulness: should be positive (MoRF > LeRF)
n_faithful = (df['c7_aopc_faithfulness'] > 0).sum()
print(f"C7 AOPC Faithfulness:")
print(f"  Mean:       {df['c7_aopc_faithfulness'].mean():.4f}")
print(f"  Std:        {df['c7_aopc_faithfulness'].std():.4f}")
print(f"  > 0 (faithful): {n_faithful}/{len(df)} storms ({n_faithful/len(df)*100:.0f}%)")

# Correlation: AOPC faithfulness vs forecast difficulty (CRPS)
corr = df['c7_aopc_faithfulness'].corr(df['c7_crps_full'])
print(f"\nCorrelation (AOPC faithfulness, forecast CRPS): {corr:.3f}")
print("(positive = harder-to-forecast cases have more faithful attributions,")
print(" consistent with XAI4Traj finding that difficult examples have cleaner segment attribution)")

---

## Appendix: ERA5 Channel Reference

| Index | Variable | Level |
|-------|----------|-------|
| 0 | Geopotential Height (GPH) | 200 hPa |
| 1 | Geopotential Height (GPH) | 500 hPa |
| 2 | Geopotential Height (GPH) | 850 hPa |
| 3 | Geopotential Height (GPH) | 925 hPa |
| 4 | U-wind | 200 hPa |
| 5 | U-wind | 500 hPa |
| 6 | U-wind | 850 hPa |
| 7 | U-wind | 925 hPa |
| 8 | V-wind | 200 hPa |
| 9 | V-wind | 500 hPa |
| 10 | V-wind | 850 hPa |
| 11 | V-wind | 925 hPa |
| 12 | SST | Surface |

## Citation

```bibtex
@article{nguyen2025tcxflow,
  title   = {TC-XFlow: Tropical Cyclone Track Forecasting via Conditional Flow Matching with Physics-Constrained ERA5 Encoder},
  author  = {Nguyen, Hong Gam and others},
  journal = {Expert Systems with Applications},
  year    = {2025}
}
```